# TSA → tomograms (tilt-series alignment with TeamTomo)

**Use this when** you have tilt images and tilt angles but **no alignment yet**. TeamTomo aligns the tilt series itself (no fiducials needed), estimates the CTF per tilt, and reconstructs. On EMPIAR-10499 series 00254 the alignment reaches 1.0 px on gold beads (IMOD's fiducial alignment: 0.73 px; Warp's AreTomo route: 2.5 px) in about 1.5 minutes.

**You need** either a Warp project folder (tilt angles and doses from `tomostar/`, images from `warp_tiltseries/tiltstack/`, CTF from Warp's full-size averages), or plain tilt stacks with a `.rawtlt`/`.tlt` file each (see `configs/tsa_to_tomograms_plain_stacks.yaml`).

**Output** in `teamtomo_output/<series>/`: the tomogram, `alignment.json` (the alignment, Å), `ctf.json` (defocus per tilt, when the averages are there), `<series>.xml` (the alignment as a Warp XML, to continue in WarpTools / M; Warp projects only) and `run.json`.

**Steps:** (1) tilt-axis angle ⇄ tiltxcorr, from a blind start; (2) projection matching, with the specimen thickness chosen from the data; (3) CTF per tilt; (4) Warp-compatible reconstruction.

In [ ]:
CONFIG = "configs/tsa_to_tomograms_warp_project.yaml"   # everything about the run is in this file (inputs, outputs, options)
SERIES = "00254"          # which tilt series to run here (batch.py runs them all)
OVERRIDES = {}            # optional, e.g. {"reconstruction": {"pixel_spacing": 5.0}}

In [ ]:
import sys, json, numpy as np, mrcfile, matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter
sys.path.insert(0, ".")
from teamtomo_workflows import load_config, discover, tsa_to_tomograms

cfg = load_config(CONFIG, OVERRIDES)
series = discover(cfg)
print(f"{cfg['workflow']}: {len(series)} tilt series found ({series[0].source if series else '-'})")
for s in series:
    print(f"  {s.name:10s} {len(s.tilt_angles)} tilts, {s.tilt_angles.min():.0f}..{s.tilt_angles.max():.0f} deg | stack: {'yes' if s.stack else 'no'} | "
          f"Warp XML: {'yes' if s.xml else 'no'} | full-size averages: {'yes' if s.averages else 'no'}")
s = next(x for x in series if x.name == SERIES)

## Align, estimate CTF, reconstruct

In [ ]:
rec = tsa_to_tomograms(cfg, s, cfg["compute"]["device"])
print(f"{rec['status']} in {rec['total_s']} s  ->  {Path(cfg['output']['root']) / s.name}")
print("steps (s):", rec["timings_s"])
print("results:", {k: v for k, v in rec["results"].items() if not isinstance(v, list) or len(v) < 4})

In [ ]:
def show_tomogram(path, title=""):
    v = mrcfile.read(path).astype(np.float32); z, y = v.shape[0] // 2, v.shape[1] // 2
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.6]})
    for a, sl, t in ((ax[0], v[z - 2:z + 3].mean(0), f"xy slice z={z}"), (ax[1], v[:, y - 2:y + 3].mean(1), f"xz slice y={y}")):
        sl = gaussian_filter(sl, 1); lo, hi = np.percentile(sl, (1, 99))
        a.imshow(sl, cmap="gray", vmin=lo, vmax=hi, origin="lower", aspect="equal" if a is ax[0] else "auto"); a.set_title(f"{title} {t}"); a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
run_dir = Path(cfg["output"]["root"]) / s.name
show_tomogram(run_dir / rec["outputs"]["tomogram"], s.name)

## What the alignment did

In [ ]:
r = rec["results"]
fig, ax = plt.subplots(1, 3, figsize=(17, 3.6))
if "thickness_scores" in r:
    t, sc = zip(*r["thickness_scores"]); ax[0].plot(t, sc, "o-"); ax[0].axvline(r["specimen_thickness_angstrom"], color="k", ls=":")
ax[0].set(title=f"specimen thickness chosen: {r['specimen_thickness_angstrom']:.0f} A", xlabel="slab thickness (A)", ylabel="image/reprojection correlation")
pm = r["projection_matching"]; ax[1].plot([p["pass"] for p in pm], [p["rms_correction_px"] for p in pm], "o-")
ax[1].set(title="projection matching: correction per pass", xlabel="pass", ylabel="rms correction (px)")
if (run_dir / "ctf.json").exists():
    ctf = json.load(open(run_dir / "ctf.json")); ok = np.array([c["reliable"] for c in ctf]); d = np.array([c["defocus_um"] for c in ctf])
    ax[2].plot(s.tilt_angles[ok], d[ok], "o", label="fit"); ax[2].plot(s.tilt_angles[~ok], d[~ok], "x", color="r", label="flagged (weak fit)")
    ax[2].set(title=f"defocus per tilt (mean {d[ok].mean():.2f} um)", xlabel="tilt angle (deg)", ylabel="defocus (um)"); ax[2].legend()
else:
    ax[2].text(0.5, 0.5, "no CTF (no full-size images)", ha="center"); ax[2].set_axis_off()
plt.tight_layout()
print(f"tilt-axis angle {r['tilt_axis_angle_deg']:.2f} deg (blind start; axis and axis+180 give mirror-image tomograms -- alignment.flip_hand)")

## All tilt series at once
```bash
python batch.py --config configs/tsa_to_tomograms_warp_project.yaml        # Warp project
python batch.py --config configs/tsa_to_tomograms_plain_stacks.yaml        # plain stacks + .rawtlt
```